# Лом: по-остра сателитна снимка със SEN2SR (ИИ)
Тетрадката взема отворената снимка на Sentinel-2 за Лом (10 м), пуска модела SEN2SR и записва снимка с около 2,5 м на пиксел.

**Важно:** детайлът, който добавя моделът, е генериран от ИИ, не е реален. На картата трябва да е означен така.
**Преди старт:** Среда за изпълнение → Промяна на типа → избери GPU (T4). Тетрадката не е изпробвана от автора, може да иска малки поправки.

In [ ]:
!pip -q install rasterio pyproj mlstac sen2sr

In [ ]:
import math, numpy as np, rasterio, torch
from rasterio.windows import Window
from pyproj import Transformer
URL = "https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/34/T/FP/2025/7/S2C_34TFP_20250726_0_L2A/"
LAT0, LON0 = 43.8167, 23.2333
KX = 111320*math.cos(math.radians(LAT0)); KY = 110574.0
# размер на изрязването в метри около центъра (по-малко = по-бързо; първо пробвай 3000 х 3000)
HALF_X, HALF_Y = 1500, 1500
t = Transformer.from_crs(4326, 32634, always_xy=True)
corners = [(-HALF_X,-HALF_Y),(HALF_X,-HALF_Y),(-HALF_X,HALF_Y),(HALF_X,HALF_Y)]
E, N = zip(*[t.transform(LON0+x/KX, LAT0+y/KY) for x,y in corners])
bands = {}
for name in ["B02","B03","B04","B08"]:
    with rasterio.open(URL+name+".tif") as ds:
        c0,r1 = ~ds.transform*(min(E),min(N)); c1,r0 = ~ds.transform*(max(E),max(N))
        c0,c1,r0,r1 = int(c0)-8,int(c1)+8,int(r0)-8,int(r1)+8
        bands[name] = ds.read(1, window=Window(c0,r0,c1-c0,r1-r0)).astype("float32")/10000.0
        origin = ds.transform*(c0,r0)
X = np.stack([bands["B02"],bands["B03"],bands["B04"],bands["B08"]])
print("форма на входа", X.shape, "стойности", X.min(), X.max())

In [ ]:
import mlstac
mlstac.download(file="https://huggingface.co/tacofoundation/sen2sr/resolve/main/SEN2SRLite/main/mlm.json", output_dir="model/SEN2SRLite")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = mlstac.load("model/SEN2SRLite").compiled_model(device=device)
print("устройство:", device)

In [ ]:
# при проблем с predict_large виж документацията на SEN2SR (github.com/ESAOpenSR/SEN2SR)
from sen2sr import predict_large
x = torch.from_numpy(X).float().to(device)
with torch.no_grad():
    sr = predict_large(model=model, X=x, overlap=32)
sr = sr.cpu().numpy()
print("изход", sr.shape)

In [ ]:
from PIL import Image
rgb = np.stack([sr[2], sr[1], sr[0]], axis=-1)           # B04,B03,B02
rgb = np.clip(rgb / 0.3, 0, 1) ** (1/1.15)                # проста яркост като при TCI
Image.fromarray((rgb*255).astype("uint8")).save("lom_sr.png")
print(rgb.shape)
from google.colab import files
files.download("lom_sr.png")

## След това
Свали `lom_sr.png` и го качи в разговора с Claude, като напишеш колко метра е изрязването (по подразбиране 3000 × 3000 м около центъра). За целия град увеличи HALF_X и HALF_Y и направи няколко изрязвания.